# 🐍 Day 6: asyncio Advanced

- gather, create_task
- Semaphores
- aiohttp
- Async generators

## Semaphore: Limit Concurrency

Limit how many coroutines run at once. Useful for rate limiting or connection pools.

In [ ]:
import asyncio

sem = asyncio.Semaphore(2)

async def limited_task(n):
    async with sem:
        print(f"Task {n} acquired semaphore")
        await asyncio.sleep(1)
        print(f"Task {n} released")

async def main():
    await asyncio.gather(limited_task(1), limited_task(2), limited_task(3), limited_task(4))

asyncio.run(main())

## wait_for: Timeout

Cancel a coroutine if it takes too long.

In [ ]:
import asyncio

async def slow():
    await asyncio.sleep(5)
    return "done"

async def main():
    try:
        result = await asyncio.wait_for(slow(), timeout=1.0)
    except asyncio.TimeoutError:
        print("Timed out!")

asyncio.run(main())

## aiohttp: Async HTTP

`pip install aiohttp`. Async HTTP client and server.

In [ ]:
import asyncio

async def fetch_url(session, url):
    async with session.get(url) as response:
        return await response.text()

async def main():
    try:
        import aiohttp
        async with aiohttp.ClientSession() as session:
            html = await fetch_url(session, "https://httpbin.org/get")
            print(f"Fetched {len(html)} bytes")
    except ImportError:
        print("Install aiohttp: pip install aiohttp")

asyncio.run(main())

## Multiple URLs with aiohttp

Fetch many URLs concurrently with a shared session.

In [ ]:
import asyncio

async def fetch_all(urls):
    try:
        import aiohttp
        async with aiohttp.ClientSession() as session:
            tasks = [session.get(url) for url in urls]
            responses = await asyncio.gather(*[asyncio.create_task(session.get(u)) for u in urls])
            results = []
            for resp in responses:
                results.append(await resp.text())
            return results
    except ImportError:
        return ["aiohttp not installed"] * len(urls)

# Simpler pattern:
async def fetch_simple(session, url):
    try:
        import aiohttp
        async with session.get(url) as resp:
            return url, await resp.text()
    except Exception as e:
        return url, str(e)

async def main():
    urls = ["https://httpbin.org/delay/0", "https://httpbin.org/delay/0"]
    try:
        import aiohttp
        async with aiohttp.ClientSession() as session:
            tasks = [fetch_simple(session, u) for u in urls]
            results = await asyncio.gather(*tasks)
            for url, content in results:
                print(f"{url}: {len(content)} bytes")
    except ImportError:
        print("pip install aiohttp")

asyncio.run(main())

## Async Generators

`async def` with `yield` produces an async generator. Use `async for` to consume.

In [ ]:
import asyncio

async def async_range(n):
    for i in range(n):
        await asyncio.sleep(0.1)
        yield i

async def main():
    async for x in async_range(5):
        print(x)

asyncio.run(main())

## run_in_executor

Run blocking (sync) code in a thread pool without blocking the event loop.

In [ ]:
import asyncio
import time

def blocking_work():
    time.sleep(1)
    return "done"

async def main():
    loop = asyncio.get_running_loop()
    result = await loop.run_in_executor(None, blocking_work)
    print(result)

asyncio.run(main())

## TaskGroup (Python 3.11+)

`asyncio.TaskGroup` provides structured concurrency: if one task fails, others are cancelled.

In [ ]:
import asyncio
import sys

if sys.version_info >= (3, 11):
    async def main():
        async with asyncio.TaskGroup() as tg:
            tg.create_task(asyncio.sleep(0.5))
            tg.create_task(asyncio.sleep(0.5))
        print("All tasks done")
    asyncio.run(main())
else:
    print("TaskGroup requires Python 3.11+")

## Common Mistakes

- **Blocking in async**: Use run_in_executor for sync code
- **Unbounded concurrency**: Use Semaphore to limit
- **Not closing aiohttp session**: Use async with
- **Mixing aiohttp with requests**: requests is sync

## Exercises

In [ ]:
# Exercise 1: Use Semaphore(2) to limit 6 tasks to 2 concurrent. Time the run.
# YOUR CODE HERE

In [ ]:
# Exercise 2: Use wait_for to timeout a slow coroutine after 0.5s.
# YOUR CODE HERE

In [ ]:
# Exercise 3: Create an async generator that yields 1, 2, 3 with 0.2s delay between each.
# YOUR CODE HERE

In [ ]:
# Exercise 4: Use run_in_executor to run a CPU-bound function (e.g., sum(range(n))).
# YOUR CODE HERE

In [ ]:
# Exercise 5: Fetch 3 URLs with aiohttp (or simulate with sleep if not installed).
# YOUR CODE HERE

In [ ]:
# Exercise 6: Implement a rate limiter: max 2 requests per second using Semaphore + sleep.
# YOUR CODE HERE

## Key Takeaways

- Semaphore: limit concurrency
- wait_for: timeout
- aiohttp: async HTTP
- run_in_executor: run sync code

## 🔜 Next: Day 7 - Mini-Project Async Scraper

Tomorrow: Build an async web scraper with aiohttp!